In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

# Path to the dataset (adjust if different)
source_dir = "/content/drive/MyDrive/project/basepaper/EuroSAT_preprocessed1"

# Check if directory exists
if os.path.exists(source_dir):
    print("✅ Dataset found at:", source_dir)
else:
    print("❌ Dataset not found. Please check the path.")


✅ Dataset found at: /content/drive/MyDrive/project/basepaper/EuroSAT_preprocessed1


In [4]:
import os
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Input preprocessed dataset
source_dir = "/content/drive/MyDrive/project/basepaper/EuroSAT_preprocessed1"
split_base = "/content/EuroSAT_full_split_3split"
train_dir = os.path.join(split_base, "train")
val_dir = os.path.join(split_base, "val")
test_dir = os.path.join(split_base, "test")

# Create output directories
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Get class names
classes = sorted(os.listdir(source_dir))

# Split images for each class
for cls in tqdm(classes, desc="Splitting Classes into Train/Val/Test"):
    cls_path = os.path.join(source_dir, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # 70% train, 15% val, 15% test
    train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

    # Create class folders
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

    # Copy images
    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(train_dir, cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(val_dir, cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(test_dir, cls, img))

print("✅ Train/Val/Test split completed at:", split_base)


Splitting Classes into Train/Val/Test: 100%|██████████| 10/10 [13:31<00:00, 81.18s/it]

✅ Train/Val/Test split completed at: /content/EuroSAT_full_split_3split


In [5]:
pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [10]:
pip install grad-cam


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 48.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44284 sha256=5d7630ba53a8bbda12bf02edafb4711871bf5d4543fd2265c09ae90f0849eb15
  Stored in directory: /root/.cache/pip/wheels/bc/52/78/893c3b94279ef238f43a9e89608af648de401b96415bebbd1f
Successfully built grad-cam


In [11]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


In [12]:
import os
import cv2
import numpy as np
from PIL import Image
import torch
import torchvision.models as models
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Define class mapping
class_names = [
    'annual_crop', 'forest', 'herbaceous_vegetation', 'highway', 'industrial',
    'pasture', 'permanent_crop', 'residential', 'river', 'sea_lake'
]
class_to_id = {name: idx for idx, name in enumerate(class_names)}

# Paths
image_root = "/content/eurosat_preprocessed"
output_root = "/content/EuroSAT_segmented"
imgsz = 224  # Resize for classifier

# Preprocessing
transform = transforms.Compose([
    transforms.Resize((imgsz, imgsz)),
    transforms.ToTensor(),
])

# Load pretrained model
model = models.resnet50(pretrained=True)
model.eval()

# Grad-CAM setup
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

# Create folders
for split in ["train", "val", "test"]:
    os.makedirs(f"{output_root}/images/{split}", exist_ok=True)
    os.makedirs(f"{output_root}/labels/{split}", exist_ok=True)

# Convert mask to YOLO-seg format
def mask_to_yolo_format(mask, class_id):
    contours, _ = cv2.findContours((mask * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    yolo_lines = []
    for contour in contours:
        if len(contour) >= 6:  # must have enough points
            contour = contour.squeeze()
            points = contour / imgsz
            flat = points.flatten()
            if len(flat) % 2 == 0:
                line = [str(class_id)] + [f"{x:.6f}" for x in flat]
                yolo_lines.append(" ".join(line))
    return yolo_lines

# Process all images
from glob import glob
import random

image_paths = []
for cls in class_names:
    image_paths += glob(f"{image_root}/{cls}/*.jpg")

random.shuffle(image_paths)
train_cutoff = int(0.7 * len(image_paths))
val_cutoff = int(0.9 * len(image_paths))

for i, img_path in enumerate(image_paths):
    label_name = os.path.basename(os.path.dirname(img_path))
    class_id = class_to_id[label_name]

    # Image prep
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0)
    rgb_img = np.array(image.resize((imgsz, imgsz))).astype(np.float32) / 255.0

    # Grad-CAM heatmap
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(class_id)])[0]
    heatmap = cv2.resize(grayscale_cam, (imgsz, imgsz))
    _, binary_mask = cv2.threshold(heatmap, 0.3, 1, cv2.THRESH_BINARY)

    # Get YOLO format mask
    label_lines = mask_to_yolo_format(binary_mask, class_id)
    if not label_lines:
        continue

    # Save image & label
    split = "train" if i < train_cutoff else "val" if i < val_cutoff else "test"
    filename = os.path.splitext(os.path.basename(img_path))[0]
    new_img_path = f"{output_root}/images/{split}/{label_name}_{i}.jpg"
    new_lbl_path = f"{output_root}/labels/{split}/{label_name}_{i}.txt"

    image.resize((imgsz, imgsz)).save(new_img_path)
    with open(new_lbl_path, "w") as f:
        f.write("\n".join(label_lines))


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:02<00:00, 39.7MB/s]


In [13]:
yaml_content = """
path: /content/EuroSAT_segmented
train: images/train
val: images/val
test: images/test

names:
  0: annual_crop
  1: forest
  2: herbaceous_vegetation
  3: highway
  4: industrial
  5: pasture
  6: permanent_crop
  7: residential
  8: river
  9: sea_lake
"""
with open("/content/EuroSAT_segmented/data.yaml", "w") as f:
    f.write(yaml_content)


In [15]:
import os
train_folder = "/content/EuroSAT_segmented/images/train"
print("Files in train folder:", os.listdir(train_folder))


Files in train folder: []


In [17]:
_, binary_mask = cv2.threshold(heatmap, 0.1, 1, cv2.THRESH_BINARY)  # lowered threshold

# Debug print
print(f"Processing {img_path} -> Label: {label_name}, Class ID: {class_id}")
print(f"Mask Non-Zero: {np.count_nonzero(binary_mask)}")

# Visualize (optional, for first image)
if i == 0:
    import matplotlib.pyplot as plt
    plt.imshow(binary_mask, cmap='gray')
    plt.title(f"Grad-CAM Binary Mask for {label_name}")
    plt.axis('off')
    plt.show()


NameError: name 'heatmap' is not defined